In [1]:
import requests 

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [2]:
import hashlib

def generate_document_id(doc):
    # combined = f"{doc['course']}-{doc['question']}"
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())
    hash_hex = hash_object.hexdigest()
    document_id = hash_hex[:8]
    return document_id

In [3]:
for doc in documents:
    doc['id'] = generate_document_id(doc)

In [4]:
documents[3]


{'text': "You don't need it. You're accepted. You can also just start learning and submitting homework without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'section': 'General course-related questions',
 'question': 'Course - I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?',
 'course': 'data-engineering-zoomcamp',
 'id': '0bbf41ec'}

In [5]:
from collections import defaultdict


In [6]:
hashes = defaultdict(list)

for doc in documents:
    doc_id = doc['id']
    hashes[doc_id].append(doc)

In [7]:
len(hashes), len(documents)


(947, 948)

In [8]:
for k, values in hashes.items():
    if len(values) > 1:
        print(k, len(values))

593f7569 2


In [9]:
hashes['593f7569']


[{'text': "They both do the same, it's just less typing from the script.\nAsked by Andrew Katoch, Added by Edidiong Esu",
  'section': '6. Decision Trees and Ensemble Learning',
  'question': 'Does it matter if we let the Python file create the server or if we run gunicorn directly?',
  'course': 'machine-learning-zoomcamp',
  'id': '593f7569'},
 {'text': "They both do the same, it's just less typing from the script.",
  'section': '6. Decision Trees and Ensemble Learning',
  'question': 'Does it matter if we let the Python file create the server or if we run gunicorn directly?',
  'course': 'machine-learning-zoomcamp',
  'id': '593f7569'}]

In [10]:
import json


In [11]:
with open('documents-with-ids.json', 'wt') as f_out:
    json.dump(documents, f_out, indent=2)

In [12]:
!head documents-with-ids.json


[
  {
    "text": "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  \u201cOffice Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon\u2019t forget to register in DataTalks.Club's Slack and join the channel.",
    "section": "General course-related questions",
    "question": "Course - When will the course start?",
    "course": "data-engineering-zoomcamp",
    "id": "c02e79ef"
  },
  {
    "text": "GitHub - DataTalksClub data-engineering-zoomcamp#prerequisites",


In [13]:
prompt_template = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record. 

The record:

section: {section}
question: {question}
answer: {text}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()


In [ ]:
import os
from mistralai.client import MistralClient



client = MistralClient()  

def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat(
        model='open-mistral-7b',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

In [19]:
from tqdm.auto import tqdm

In [20]:
results = {}


In [21]:
for doc in tqdm(documents): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions = generate_questions(doc)
    results[doc_id] = questions

  0%|          | 0/948 [00:00<?, ?it/s]

In [22]:
result={}

In [23]:
import pickle

In [38]:
import re


In [29]:
results['1f6520ca']


'[\n  "question1: What are the prerequisites for this course?",\n  "question2: What skills or prior knowledge are required to enroll in this course?",\n  "question3: What are the necessary prerequisites for the Data Engineering Zoomcamp course offered by DataTalksClub on GitHub?",\n  "question4: What should a student know before starting the Data Engineering Zoomcamp course offered by DataTalksClub on GitHub?",\n  "question5: What are the prerequisites for the Data Engineering course provided by DataTalksClub on GitHub?"\n]'

In [39]:
def escape_backslashes(s):
    return re.sub(r'(?<!\\)\\(?![\\nrt"\'/bfu])', r'\\\\', s)

parsed_results = {}
for key, value in results.items():
    try:
        safe_value = escape_backslashes(value)
        parsed_results[key] = json.loads(safe_value)
    except json.JSONDecodeError as e:
        print(f"Error parsing key: {key} — {e}")


Error parsing key: c02e79ef — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: 7842b56a — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: 0bbf41ec — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: 63394d91 — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: 93e2c8ed — Expecting ',' delimiter: line 2 column 12 (char 13)
Error parsing key: a482086d — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: 47972cb1 — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: ddf6c1b3 — Expecting ',' delimiter: line 2 column 12 (char 13)
Error parsing key: 3c0114ce — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: d061525d — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: d407d65b — Expecting ',' delimiter: line 2 column 14 (char 15)
Error parsing key: e866156b — Expecting ',' delimiter: line 5 column 175 (char 761)
Error parsing 

In [40]:
doc_index = {d['id']: d for d in documents}

In [41]:
final_results = []

for doc_id, questions in parsed_resulst.items():
    course = doc_index[doc_id]['course']
    for q in questions:
        final_results.append((q, course, doc_id))

In [42]:
import pandas as pd

In [43]:
df = pd.DataFrame(final_results, columns=['question', 'course', 'document'])


In [44]:
!wget https://github.com/DataTalksClub/llm-zoomcamp/blob/main/03-evaluation/search_evaluation/ground-truth-data.csv

--2025-10-08 17:51:58--  https://github.com/DataTalksClub/llm-zoomcamp/blob/main/03-evaluation/search_evaluation/ground-truth-data.csv
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘ground-truth-data.csv’

ground-truth-data.c     [  <=>               ] 711.85K  3.41MB/s    in 0.2s    

2025-10-08 17:51:59 (3.41 MB/s) - ‘ground-truth-data.csv’ saved [728939]



In [45]:
df.to_csv('ground-truth-data.csv', index=False)


In [46]:
!head ground-truth-data.csv


question,course,document
question1: What are the prerequisites for this course?,data-engineering-zoomcamp,1f6520ca
question2: What skills or prior knowledge are required to enroll in this course?,data-engineering-zoomcamp,1f6520ca
question3: What are the necessary prerequisites for the Data Engineering Zoomcamp course offered by DataTalksClub on GitHub?,data-engineering-zoomcamp,1f6520ca
question4: What should a student know before starting the Data Engineering Zoomcamp course offered by DataTalksClub on GitHub?,data-engineering-zoomcamp,1f6520ca
question5: What are the prerequisites for the Data Engineering course provided by DataTalksClub on GitHub?,data-engineering-zoomcamp,1f6520ca
What is the schedule for the Data-Engineering ZoomCamp?,data-engineering-zoomcamp,2ed9b986
Are there multiple cohorts for each ZoomCamp within a year?,data-engineering-zoomcamp,2ed9b986
How many ZoomCamps are available for the Machine Learning course in a year?,data-engineering-zoomcamp,2ed9b986
Can I ta